# 01: Explore Data and Feature Engineering (Task 1)

A thin interactive wrapper around `src.data_fetching` and `src.features`. All the logic lives in `src/`. This notebook just calls it and shows the results.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_fetching import load_or_fetch_smiles
from src.features import (
    build_descriptor_table, build_modeling_table, get_candidate_matrix,
    compute_feature_correlations, select_k_best_features, get_modeling_features,
)
from src.config import CLASSIFICATION_TARGET, DESCRIPTORS_CSV

In [ ]:
smiles_df = load_or_fetch_smiles()
descriptors = pd.read_csv(DESCRIPTORS_CSV) if DESCRIPTORS_CSV.exists() else build_descriptor_table(smiles_df)
data = build_modeling_table(descriptors)
data.head()

In [ ]:
corr = compute_feature_correlations(data)
corr

In [ ]:
X = get_candidate_matrix(data)
y = data[CLASSIFICATION_TARGET]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pd.concat([X, y], axis=1).corr(), cmap="RdBu_r", center=0, ax=ax)
plt.show()

### Automated selection (for reference)

The table below is the fold-safe `SelectKBest` sweep. This is what runs by default.

In [ ]:
selection = select_k_best_features(X, y)
print("best_k:", selection["best_k"])
print("scores_by_k:", selection["scores_by_k"])
print("selected_features:", selection["selected_features"])

### Modeling feature set actually used downstream

`get_modeling_features` is what `main.py` and the other notebooks call: it honors a manual
`MANUAL_FEATURES` override in `.env` (e.g. after eyeballing the heatmap above) and otherwise
falls back to the automated selection printed above. Check `selection["source"]` to see which
path was taken.

In [ ]:
selection = get_modeling_features(X, y)
print("source:", selection["source"])
print("selected_features:", selection["selected_features"])
print("features_by_k[6] (used by the fixed 6-qubit QCNN):", selection["features_by_k"][6])